# Приоритизация обращений

Решение основано на CatBoost. Готовые признаки из `train.csv` и `test.csv` дополнены историей из `events.csv`, для каждого обращения рассчитаны активность, давность событий, последние действия и динамика цены. Использовались только события с `event_ts < assignment_ts`, поэтому информация из будущего в модель не попадает.

Валидация построена по времени: гипотезы проверялись на последовательных временных фолдах, а последние четыре даты оставались под финальную проверку. Дополнительные отношения признаков, сокращение набора, другие модели и ансамбли не дали стабильного прироста. Поэтому финальное решение использует один CatBoost, обученный на всём `train`. Вероятность положительного класса сохраняется как `score`.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier

ROOT = Path(".")
DATA_DIR = ROOT / "data"
TARGET = "target"
RANDOM_STATE = 42
MODEL_ITERATIONS = 800


C:\Users\senni\AppData\Roaming\Python\Python39\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\senni\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:62: UserWarning: Pandas requires version '1.3.4' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


## Загрузка данных

Загружаем train, test, события.


In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
events = pd.read_csv(DATA_DIR / "events.csv")

print("train:", train.shape)
print("test:", test.shape)
print("events:", events.shape)
print("target mean:", round(train[TARGET].mean(), 4))


train: (13694, 119)
test: (4306, 118)
events: (254705, 7)
target mean: 0.2075


Доля положительного класса 20,75%, целевая переменная несбалансирована, что учитывается при оценке модели через Daily AP.

## Признаки из событий

Дополняем готовые табличные признаки информацией из `events.csv`. Для каждого обращения используем только события с `event_ts < assignment_ts`, то есть доступные модели на момент назначения, чтобы исключить попадание в признаки информации из будущего.

Из истории событий формируем:

- общую активность и разнообразие действий
- давность, частоту и регулярность событий
- активность в нескольких временных окнах
- последовательность последних действий
- уровень и динамику цены

In [3]:
def build_event_features(leads: pd.DataFrame, events_df: pd.DataFrame) -> pd.DataFrame:
    """
    Преобразует историю событий в одну строку признаков для каждого обращения.
    Использует только события до момента назначения и описывает активность,
    давность и последовательность действий, временные окна и динамику цены.
    """
    # оставляем время назначения и текущую цену объявления
    context = leads[["lead_id", "assignment_ts", "item_price_log"]].copy()
    context["assignment_ts"] = pd.to_datetime(context["assignment_ts"])

    # привязываем события к обращениям как одна строка lead может иметь много событий
    history = events_df.merge(
        context,
        on="lead_id",
        how="inner",
        suffixes=("_event", "_lead"),
        validate="many_to_one",
    )
    history["event_ts"] = pd.to_datetime(history["event_ts"])

    # события после назначения модели недоступны
    history = history[history["event_ts"] < history["assignment_ts"]].copy()

    # переводим timestamp в понятные признаки давности и календарного контекста
    history["hours_before_assignment"] = (
        history["assignment_ts"] - history["event_ts"]
    ).dt.total_seconds() / 3600
    history["event_day"] = history["event_ts"].dt.floor("D")
    history["event_hour"] = history["event_ts"].dt.hour
    history["event_weekend"] = (history["event_ts"].dt.weekday >= 5).astype(int)
    history["event_type_ctx"] = history["event_type"] + "_" + history["ctx_seq"]
    history = history.sort_values(["lead_id", "event_ts"])
    history["event_price_step"] = history.groupby("lead_id")["item_price_log_event"].diff()

    # базовые характеристики всей доступной истории обращения
    grouped = history.groupby("lead_id", sort=False)
    features = grouped.agg(
        event_count=("event_type", "size"),
        event_active_days=("event_day", "nunique"),
        event_type_count=("event_type", "nunique"),
        event_ctx_count=("ctx_seq", "nunique"),
        event_slot_count=("src_slot", "nunique"),
        event_recency_hours=("hours_before_assignment", "min"),
        event_history_hours=("hours_before_assignment", "max"),
        event_price_mean=("item_price_log_event", "mean"),
        event_price_std=("item_price_log_event", "std"),
        event_price_min=("item_price_log_event", "min"),
        event_price_max=("item_price_log_event", "max"),
        event_price_first=("item_price_log_event", "first"),
        event_price_last=("item_price_log_event", "last"),
        event_price_median=("item_price_log_event", "median"),
        event_price_unique=("item_price_log_event", "nunique"),
        event_price_step_mean=("event_price_step", "mean"),
        event_price_step_std=("event_price_step", "std"),
        event_price_step_min=("event_price_step", "min"),
        event_price_step_max=("event_price_step", "max"),
        event_price_step_last=("event_price_step", "last"),
        event_slot_mean=("src_slot", "mean"),
        event_slot_std=("src_slot", "std"),
        event_slot_min=("src_slot", "min"),
        event_slot_max=("src_slot", "max"),
        event_hour_mean=("event_hour", "mean"),
        event_hour_std=("event_hour", "std"),
        last_event_hour=("event_hour", "last"),
        event_weekend_share=("event_weekend", "mean"),
    )

    # квантили дополняют среднее и std
    # они устойчивее к редким скачкам цены
    quantiles = grouped["item_price_log_event"].quantile([0.1, 0.25, 0.75, 0.9]).unstack()
    quantiles.columns = ["event_price_q10", "event_price_q25", "event_price_q75", "event_price_q90"]
    features = features.join(quantiles, how="left")

    features["event_price_range"] = features["event_price_max"] - features["event_price_min"]
    features["event_price_iqr"] = features["event_price_q75"] - features["event_price_q25"]
    features["event_price_change"] = features["event_price_last"] - features["event_price_first"]
    features["event_price_change_per_day"] = features["event_price_change"] / (
        features["event_history_hours"] / 24 + 1
    )

    # oтдельно описываем направление и размер изменений цены между событиями
    price_steps = history.assign(
        absolute_step=history["event_price_step"].abs(),
        positive_step=(history["event_price_step"] > 0).astype(float),
    )
    features["event_price_step_abs_mean"] = price_steps.groupby("lead_id")["absolute_step"].mean()
    features["event_price_step_positive_share"] = price_steps.groupby("lead_id")["positive_step"].mean()
    features["events_per_active_day"] = features["event_count"] / features["event_active_days"]
    features["events_per_history_day"] = features["event_count"] / (
        features["event_history_hours"] / 24 + 1
    )

    # считаем, сколько раз встречался каждый тип события, контекст и слот
    for column, prefix in [
        ("event_type", "event_type"),
        ("ctx_seq", "event_ctx"),
        ("src_slot", "event_slot"),
    ]:
        counts = pd.crosstab(history["lead_id"], history[column]).add_prefix(f"{prefix}_count_")
        features = features.join(counts, how="left")

    # совместная пара type u context сохраняет больше информации, чем два признака отдельно
    type_ctx_counts = pd.crosstab(history["lead_id"], history["event_type_ctx"]).add_prefix(
        "event_type_ctx_count_"
    )
    features = features.join(type_ctx_counts, how="left")

    # окна показывают, насколько активно пользователь вёл себя прямо перед назначением
    for hours in [6, 12, 24, 72, 168, 336, 720]:
        recent = history[history["hours_before_assignment"] <= hours]
        total = recent.groupby("lead_id").size().rename(f"events_{hours}h")
        type_counts = pd.crosstab(recent["lead_id"], recent["event_type"]).add_prefix(
            f"event_type_{hours}h_"
        )
        features = features.join(total, how="left").join(type_counts, how="left")

        if hours in [24, 72, 168]:
            price_stats = recent.groupby("lead_id")["item_price_log_event"].agg(
                ["mean", "std", "min", "max"]
            )
            price_stats.columns = [f"event_price_{hours}h_{name}" for name in price_stats.columns]
            features = features.join(price_stats, how="left")

    # взвешенные счётчики плавно уменьшают вклад старых событий без жёсткой границы окна
    for half_life_days in [1, 3, 7, 14]:
        weight_column = f"weight_{half_life_days}d"
        history[weight_column] = np.exp(
            -np.log(2) * history["hours_before_assignment"] / (24 * half_life_days)
        )
        weighted_total = history.groupby("lead_id")[weight_column].sum().rename(
            f"weighted_events_{half_life_days}d"
        )
        weighted_types = history.pivot_table(
            index="lead_id",
            columns="event_type",
            values=weight_column,
            aggfunc="sum",
        ).add_prefix(f"weighted_event_type_{half_life_days}d_")
        features = features.join(weighted_total, how="left").join(weighted_types, how="left")

    # для каждого типа и контекста считаем время с последнего такого события
    type_recency = history.pivot_table(
        index="lead_id",
        columns="event_type",
        values="hours_before_assignment",
        aggfunc="min",
    ).add_prefix("event_type_recency_hours_")
    ctx_recency = history.pivot_table(
        index="lead_id",
        columns="ctx_seq",
        values="hours_before_assignment",
        aggfunc="min",
    ).add_prefix("event_ctx_recency_hours_")
    features = features.join(type_recency, how="left").join(ctx_recency, how="left")

    # интервалы между событиями описывают плотность и регулярность активности
    history["gap_hours"] = grouped["event_ts"].diff().dt.total_seconds() / 3600
    gaps = history.groupby("lead_id")["gap_hours"].agg(["mean", "std", "min", "max", "median", "last"])
    gaps.columns = [f"event_gap_hours_{name}" for name in gaps.columns]
    features = features.join(gaps, how="left")

    # последние пять событий сохраняем отдельно
    # ближайшие действия обычно самые информативные
    sequence = history.groupby("lead_id", sort=False).tail(5).copy()
    sequence["position"] = sequence.groupby("lead_id").cumcount(ascending=False) + 1
    for column, prefix in [
        ("event_type", "last_event_type"),
        ("ctx_seq", "last_event_ctx"),
        ("src_slot", "last_event_slot"),
        ("event_type_ctx", "last_event_type_ctx"),
    ]:
        values = sequence.pivot(index="lead_id", columns="position", values=column)
        values.columns = [f"{prefix}_{int(position)}" for position in values.columns]
        features = features.join(values, how="left")

    # сравниваем последнюю цену в истории с ценой на момент назначения
    features = features.join(context.set_index("lead_id")["item_price_log"], how="left")
    features["event_price_vs_lead"] = features["event_price_last"] - features["item_price_log"]
    return features.drop(columns="item_price_log").reset_index()

## Подготовка данных для модели

Объединяем `train` и `test` только для формирования одинакового набора признаков. Целевая переменная не используется, для каждой строки событийные признаки рассчитываются отдельно по её собственной истории.

После добавления событийных признаков исключаем идентификаторы, даты, служебные поля и `target`. Категориальные и небольшие дискретные признаки передаём CatBoost как категории, а пропуски в них заменяем отдельным значением. Затем возвращаем исходное разделение на обучающую и тестовую выборки.

In [4]:
# объединяем train и test только для одинаковой схемы признаков
train_with_part = pd.concat(
    [train, pd.DataFrame({"data_part": "train"}, index=train.index)],
    axis=1,
)
test_with_part = pd.concat(
    [test, pd.DataFrame({"data_part": "test", "target": np.nan}, index=test.index)],
    axis=1,
)
all_leads = pd.concat([train_with_part, test_with_part], ignore_index=True, sort=False)

# для каждой строки строим признаки только из её собственной истории событий
event_features = build_event_features(all_leads, events)
all_data = all_leads.merge(
    event_features,
    on="lead_id",
    how="left",
    sort=False,
    validate="one_to_one",
)

# исключаем идентификаторы, даты, служебные поля и target
NON_FEATURE_COLUMNS = {
    "lead_id", "user_id", "assignment_ts", "assignment_date",
    "target", "split", "data_part",}
feature_columns = [column for column in all_data.columns if column not in NON_FEATURE_COLUMNS]

# CatBoost умеет работать с категориями, напрямую передаём их отдельным списком
categorical_columns = [column for column in feature_columns if not pd.api.types.is_numeric_dtype(all_data[column])]

# небольшие дискретные значения трактуем как категории, а не как непрерывную шкалу
discrete_columns = [
    "assignment_hour", "assignment_weekday", "is_weekend", "prior_assignments_30d",
    "event_count", "event_active_days", "event_type_count", "event_ctx_count",
    "event_slot_count", "event_slot_min", "event_slot_max",
]
discrete_columns += [column for column in feature_columns if column.startswith("last_event_slot_")]

categorical_columns += [column for column in discrete_columns if column not in categorical_columns]

# заполняем пропуски
all_data[discrete_columns] = (all_data[discrete_columns].fillna(-1).round().astype(int).astype(str))

all_data[categorical_columns] = (all_data[categorical_columns].fillna("__missing__").astype(str))

all_data = all_data.copy()

# возвращаем исходное разделение после полностью одинаковой обработки
train_data = all_data[all_data["data_part"] == "train"].copy()
test_data = all_data[all_data["data_part"] == "test"].copy()

print("all features:", len(feature_columns))
print("categorical features:", len(categorical_columns))

all features: 350
categorical features: 38


Так сформировано 350 признаков, из которых 38 передаются модели как категориальные. Одинаковая обработка `train` и `test` гарантирует совпадение структуры данных при обучении.

## Метрика и модель

Используем Daily Average Precision. Для обучения используем CatBoost, он напрямую работает с категориальными признаками и числовыми пропусками. Параметры и `random_seed` фиксируем, так валидация и итоговое обучение будут полностью воспроизводимыми.

In [5]:
def daily_average_precision(y_true, scores, dates) -> float:
    """
    Считает Average Precision отдельно для каждой даты назначения
    и возвращает среднее значение по всем дням.
    """
    metric_data = pd.DataFrame(
        {
            "target": np.asarray(y_true),
            "score": np.asarray(scores),
            "date": pd.to_datetime(dates).dt.date,
        }
    )
    daily_scores = [
        average_precision_score(day["target"], day["score"])
        for _, day in metric_data.groupby("date")
    ]
    return float(np.mean(daily_scores))


def make_model():
    """
    Создаёт CatBoostClassifier с фиксированными параметрами.
    """
    return CatBoostClassifier(
        iterations=MODEL_ITERATIONS,
        depth=6,
        learning_rate=0.03,
        l2_leaf_reg=7,
        random_strength=0.3,
        boosting_type="Ordered",
        loss_function="Logloss",
        random_seed=RANDOM_STATE,
        max_ctr_complexity=2,
        one_hot_max_size=10,
        verbose=False,
        allow_writing_files=False,
    )


## Валидация

Так как тестовые данные относятся к более позднему периоду, используем временное разделение. Модель обучается на первых датах `train`, а последние четыре даты оставляются для итоговой проверки.

Валидационные строки не участвуют в обучении, а параметры модели зафиксированы заранее.

In [6]:
assignment_dates = pd.to_datetime(train_data["assignment_date"])
ordered_dates = sorted(assignment_dates.unique())
validation_dates = ordered_dates[-4:]

fit_mask = ~assignment_dates.isin(validation_dates)
validation_mask = assignment_dates.isin(validation_dates)

validation_model = make_model()
validation_model.fit(
    train_data.loc[fit_mask, feature_columns],
    train_data.loc[fit_mask, TARGET].astype(int),
    cat_features=categorical_columns,
)

validation_scores = validation_model.predict_proba(
    train_data.loc[validation_mask, feature_columns]
)[:, 1]
validation_daily_ap = daily_average_precision(
    train_data.loc[validation_mask, TARGET].astype(int),
    validation_scores,
    train_data.loc[validation_mask, "assignment_date"],
)

print(f"Validation Daily AP: {validation_daily_ap:.5f}")


Validation Daily AP: 0.73912


## Submission

Обучаем ту же модель на всём train и сохраняем вероятность положительного класса в колонку `score`.


In [7]:
final_model = make_model()
final_model.fit(
    train_data[feature_columns],
    train_data[TARGET].astype(int),
    cat_features=categorical_columns,
)

test_scores = final_model.predict_proba(test_data[feature_columns])[:, 1]

submission = pd.DataFrame(
    {
        "lead_id": test["lead_id"].astype(str),
        "score": test_scores,
    }
)

assert len(submission) == len(test)
assert submission["lead_id"].is_unique
assert submission["score"].notna().all()
assert submission["score"].between(0, 1).all()

submission.to_csv(ROOT / "submission.csv", index=False)
submission.head()

,lead_id,score
0,lead_97e409eb8f8c8246,0.008578
1,lead_55310edb4489f9e9,0.135585
2,lead_e7f653a2c6a7eee8,0.627931
3,lead_22f8e1cfc487ac20,0.062933
4,lead_48b638b839abfac3,0.083555
